# Combine NQM Flat Files

Use this notebook to combine multiple monthly NQM pool flat files into one tab-delimited flat file with a single header row.

The default settings combine the ten pools from the ENU top-10 request into `NQM_TOP10_20260401.txt` under `N:/FlatFilesMonthly/NQMMonthly/`.

In [ ]:
from pathlib import Path

# Edit these parameters as needed.
source_dir = Path(r"N:/FlatFilesMonthly/NQMMonthly")
asof_date = "20260401"
output_pool_id = "NQM_TOP10"
output_path = source_dir / f"{output_pool_id}_{asof_date}.txt"

pool_ids = [
    "B3A",  # VERUS 2022-1
    "ENU",  # VISIO 2023-1
    "ESU",  # CHNGE 2023-4
    "B4O",  # COLT 2022-3
    "B4U",  # VERUS 2022-3
    "ES8",  # ADMT 2023-NQM3
    "EVD",  # ADMT 2024-NQM1
    "B9U",  # CHNGE 2022-1
    "B9V",  # CHNGE 2022-2
    "Q4Y",  # BRAVO 2021-NQM1
]

source_files = [source_dir / f"{pool_id}_{asof_date}.txt" for pool_id in pool_ids]
source_files

In [ ]:
def count_lines(path: Path) -> int:
    with path.open("rb") as f:
        return sum(1 for _ in f)

missing = [path for path in source_files if not path.exists()]
if missing:
    raise FileNotFoundError("Missing source files:\n" + "\n".join(str(path) for path in missing))

for path in source_files:
    print(f"{path.name}: {count_lines(path) - 1:,} data rows")

In [ ]:
def combine_flatfiles(source_files: list[Path], output_path: Path) -> int:
    """Combine tab-delimited pool flat files with one shared header."""
    header = None
    data_rows = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8", newline="") as out_file:
        for path in source_files:
            with path.open("r", encoding="utf-8", newline="") as in_file:
                current_header = in_file.readline()
                if not current_header:
                    raise ValueError(f"Empty source file: {path}")

                if header is None:
                    header = current_header
                    out_file.write(header)
                elif current_header != header:
                    raise ValueError(f"Header mismatch in {path}")

                for line in in_file:
                    out_file.write(line)
                    data_rows += 1

    return data_rows

rows_written = combine_flatfiles(source_files, output_path)
print(f"Wrote {rows_written:,} rows to {output_path}")

In [ ]:
# Validate the combined file row count.
expected_rows = sum(count_lines(path) - 1 for path in source_files)
actual_rows = count_lines(output_path) - 1

print(f"Expected rows: {expected_rows:,}")
print(f"Actual rows:   {actual_rows:,}")
assert actual_rows == expected_rows

print("Combined flat file is ready:")
print(output_path)

## Run LMSim2

Use `LMSim2/test_config_nqm_top10.json` in the LMSim repo to run this combined flat file. That config expects the combined file name to be `NQM_TOP10_20260401.txt` in `N:/FlatFilesMonthly/NQMMonthly/`.